# Butterfly shimmer FDN: a minimal demo

How to use `flyshim`: build the network, run audio through it, set its parameters, and change them over time. Each example is shown as a spectrogram, played below its cell, and saved as a WAV file in `notebooks/output/` (ignored by git). How the network works is explained in the [README](../README.md).

The input is [`clips/dry.wav`](../clips/dry.wav): four phrases of three short, descending notes between about 480 and 1280 Hz, one phrase every 2.2 s. Each render takes a few seconds.

In [ ]:
import sys
from pathlib import Path

# Run from notebooks/ or from the repository root. flyshim and pyFDN are imported from
# this checkout (src/ and the pyFDN submodule), whether or not they are installed.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "pyFDN" / "src")]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from matplotlib.ticker import EngFormatter
from scipy.signal import spectrogram

import flyshim as fly

dry, fs = sf.read(ROOT / "clips" / "dry.wav")  # mono, 48 kHz
x = np.concatenate([dry, np.zeros(6 * fs)])  # 6 s of silence for the tail

OUT = ROOT / "notebooks" / "output"
OUT.mkdir(exist_ok=True)


def spectrograms(signals, marks=(), fmin=50, fmax=20000, dyn=70):
    """One spectrogram per signal, channels power-averaged, log-frequency axis."""
    fig = plt.figure(figsize=(3 * len(signals) + 1, 2.6), layout="constrained")
    axes = fig.subplots(1, len(signals), sharey=True, squeeze=False)
    for ax, (title, y) in zip(axes.flat, signals.items(), strict=True):
        f, t, S = spectrogram(np.atleast_2d(y.T), fs, nperseg=4096, noverlap=3072)
        P = S.mean(axis=0)[1:]  # without 0 Hz, which a log axis cannot show
        level = 10 * np.log10(P / P.max() + 1e-12)  # dB re peak
        mesh = ax.pcolormesh(t, f[1:], level, vmin=-dyn, vmax=0, cmap="Blues")
        for t_mark in marks:
            ax.axvline(t_mark, color="#eb6834", ls="--", lw=1.2)
        ax.set(title=title, xlabel="Time (s)", yscale="log", ylim=(fmin, fmax))
        ax.yaxis.set_major_formatter(EngFormatter(unit="Hz"))
    fig.colorbar(mesh, ax=axes, label="dB re peak")
    plt.show()


def listen(examples):
    """Play each example and save it to output/<name>.wav, at a peak of -1 dBFS."""
    # the last 0.5 s fade out, so that tails cut off by the end do not stop abruptly
    fade = np.linspace(1, 0, fs // 2)[:, None]
    for name, y in examples.items():
        y = 10 ** (-1 / 20) * y.reshape(len(y), -1) / np.abs(y).max()  # (T, channels)
        y[-len(fade) :] *= fade
        sf.write(OUT / f"{name}.wav", y, fs, subtype="PCM_24")
        print(f"{name}.wav")
        display(Audio(y.T, rate=fs))

## 1. Run a clip through the network

`ButterflyShimmerFDN(fs)` builds the network with every parameter at its default, and `process_signal` runs a whole signal through it: mono `(T,)` or stereo `(T, 2)` in, stereo `(T, 2)` out, wet only. With the defaults, every pass through FDN<sub>S</sub> transposes by an octave, the tones keep climbing for about 3 s (the shift time), and FDN<sub>H</sub> rings with a T60 of 8 s.

In [ ]:
net = fly.ButterflyShimmerFDN(fs)  # every parameter at its default
examples = {"dry": x, "default": net.process_signal(x)}
spectrograms(examples)
listen(examples)

All parameters and their defaults are in `fly.DEFAULTS` (angles in radians). Any of them can be passed to the constructor, to `set` (section 5), or in the events of `render` (section 4). [Usage](../README.md#usage) in the README lists their symbols and meanings.

In [ ]:
fly.DEFAULTS

## 2. Shift time: how long the tones climb

A 0.2 s sine burst at 220 Hz makes the climb easy to see. Every pass through FDN<sub>S</sub> adds a tone an octave above the previous one, and hands part of the energy over to FDN<sub>H</sub>, which sustains it. `shift_time` ($T_\text{S}$) is the time in which FDN<sub>S</sub> decays by 60 dB:

- `0`: no climb, only the conventional reverb of FDN<sub>H</sub>;
- `1` s: a few octaves;
- `10` s: octaves up to the top of the band, which keep ringing in FDN<sub>H</sub>;
- `np.inf`: the classic shimmer FDN. Nothing is handed over, so the tones climb to the top of the band within a second and are gone.

In [ ]:
n = int(0.2 * fs)
burst = 0.3 * np.sin(2 * np.pi * 220 * np.arange(n) / fs) * np.hanning(n)
probe = np.concatenate([burst, np.zeros(6 * fs)])

climbs = {}
for label, T in [("0s", 0.0), ("1s", 1.0), ("10s", 10.0), ("inf", np.inf)]:
    net = fly.ButterflyShimmerFDN(fs, shift_time=T)
    climbs[f"shift_time_{label}"] = net.process_signal(probe)
spectrograms(climbs)
listen(climbs)

## 3. A different interval on each side

`transposition` sets the interval per pass in semitones, `(left, right)`, or one number for both. With the default `stereo_coupling_S = 0`, each channel of FDN<sub>S</sub> climbs its own ladder. Here `stereo_coupling_H` is set to 0 as well, so that FDN<sub>H</sub> does not spread the tail across the channels (at its default, π/8, it does), and the burst climbs in octaves on the left and in fifths on the right. Raising `stereo_coupling_S` towards π/2 sends tones across to the other channel, so the intervals combine (+12 then +7 = +19 semitones).

In [ ]:
# +12 st on the left, +7 st on the right, and the two sides kept apart in FDN_H too
net = fly.ButterflyShimmerFDN(fs, transposition=(12, 7), stereo_coupling_H=0)
y = net.process_signal(probe)
spectrograms({"left: +12 st": y[:, 0], "right: +7 st": y[:, 1]})

net.reset()  # empty the network, keep the parameters
listen({"transposition_12_7": net.process_signal(x)})

## 4. Hold, and scheduled changes

`fly.render(net, x, events)` runs `x` through the network and changes parameters at given times, with `events = [(time in s, {name: value, ...}), ...]`. Changes are smoothed, so they do not click.

Here, the hold amount (`hold_amount`, $a_\text{H}$) goes to 1 after the first phrase. The hold half of FDN<sub>H</sub> is then closed off and sustains what it holds indefinitely, while the next three phrases climb over it and decay in the open half. At 9 s the hold is released and the reverberation time is halved, so that everything dies away in about 4 s. The dashed lines mark the two events.

In [ ]:
events = [
    (1.2, {"hold_amount": 1}),  # hold, after the first phrase
    (9.0, {"hold_amount": 0, "t60_dc": 4.0, "t60_ny": 2.0}),  # release, decay faster
]
held = fly.render(fly.ButterflyShimmerFDN(fs), x, events)
spectrograms({"hold": held}, marks=[t for t, _ in events])
listen({"hold": held})

## 5. Live changes and stored energy

The network keeps its state between calls, so it can also run block by block, as in a real-time host, with `set` changing parameters in between (smoothly, unless `snap=True`). `stored_energy` returns the energy in FDN<sub>S</sub> (`"S"`) and in the open (`"O"`) and hold (`"H"`) halves of FDN<sub>H</sub>. Once the hold is engaged, the hold half is lossless: over the next 10 s its energy does not change, while the open half decays with the T60 of 8 s and FDN<sub>S</sub> empties.

In [ ]:
net = fly.ButterflyShimmerFDN(fs)
net.process_signal(dry[: 2 * fs])  # the first phrase
net.set(hold_amount=1)  # reached along a 0.5 s ramp
net.process_signal(np.zeros(fs))
before = net.stored_energy()
net.process_signal(np.zeros(10 * fs))  # 10 s later
after = net.stored_energy()
for part, name in [("S", "FDN_S"), ("O", "open half"), ("H", "hold half")]:
    print(f"{name:9}  {10 * np.log10(after[part] / before[part]):+8.2f} dB in 10 s")

## Next

- [Usage](../README.md#usage) in the README lists every parameter with its symbol and meaning. The constructor also takes settings that are fixed at creation, such as the number of delay lines (`n_shift`, `n_hold`), their lengths (`shift_pass`, `hold_delays`), and the `seed`.
- Things to try: `transposition=-12`, so that the tones climb down; `transposition=(12, 7)` with `stereo_coupling_S=np.pi / 4`; or the hold with `mod_depth=0`, so that the held sound stops changing.